In [ ]:
"""MyoSeg: two-dimensional muscle fiber segmentation and morphometry.

Supports single-channel fluorescence and original RGB H&E microscopy images.
Adapted from MyoSeg_v1.0.0.ipynb. All user input comes from the GUI.
Cellpose 3 is imported only when segmentation is requested.

Launch from the shared MyoTools environment:
    conda activate myotools
    voila MyoSeg/MyoSeg_v1.1.1.ipynb

Other dependencies already used by the source notebook: numpy, pandas,
scipy, scikit-image, tifffile, Pillow, torch and cellpose 3.x.
CZI input additionally requires aicsimageio and aicspylibczi.
Validate masks and measurements on representative real tissue before routine use.
"""
import asyncio
import hashlib
import html
import io
import json
import math
import os
import re
import shutil
import threading
import traceback
import uuid
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile as tiff
from scipy.ndimage import binary_erosion
from scipy.spatial import ConvexHull, QhullError
from skimage.measure import regionprops
from skimage.segmentation import find_boundaries
from skimage.transform import resize
from PIL import Image

APP_VERSION = "1.1.1"
DETAIL_COLUMNS = ["Image_ID", "Filename", "Scene", "Region_Label", "Area_µm²",
                  "Feret_Diameter_µm", "Touches_border", "Included", "Exclusion_reason"]


def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return "not installed"


def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def signature(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     allow_nan=False).encode("utf-8")).hexdigest()


def file_stamp(path):
    path = Path(path).resolve()
    info = path.stat()
    return {"path": str(path), "size": info.st_size, "mtime_ns": info.st_mtime_ns}


def inside(path, parent):
    try:
        Path(path).resolve().relative_to(Path(parent).resolve())
        return True
    except ValueError:
        return False


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(f".{path.stem}.{uuid.uuid4().hex}.tmp")
    try:
        temp.write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                   allow_nan=False), encoding="utf-8")
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)


def atomic_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(f".{path.stem}.{uuid.uuid4().hex}.csv")
    try:
        frame.to_csv(temp, index=False, encoding="utf-8-sig")
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)


def write_image(path, array, spacing):
    """Write 2D grayscale or YXS RGB OME-TIFF with calibration in µm."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(f".{path.stem}.{uuid.uuid4().hex}.tif")
    try:
        if array.ndim == 2:
            axes, photometric = "YX", "minisblack"
        elif array.ndim == 3 and array.shape[-1] == 3:
            axes, photometric = "YXS", "rgb"
        else:
            raise ValueError(f"Cannot write non-grayscale/non-RGB image of shape {array.shape}.")
        tiff.imwrite(temp, array, ome=True, photometric=photometric,
                     bigtiff=array.nbytes >= 2**32 - 2**25,
                     metadata={"axes": axes, "PhysicalSizeY": spacing[0],
                               "PhysicalSizeX": spacing[1],
                               "PhysicalSizeYUnit": "µm", "PhysicalSizeXUnit": "µm"})
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)


def validate_settings(settings, require_model=True):
    s = dict(settings)
    # Previous notebooks and stored scripts can still use the original 2D mode.
    s.setdefault("image_format", "single")
    s.setdefault("rgb_model_channels", "two")
    s.setdefault("cellpose_primary", 0)
    s.setdefault("cellpose_secondary", 0)
    if s["image_format"] not in ("single", "rgb"):
        raise ValueError("Image type must be Single-channel or RGB brightfield.")
    if s["rgb_model_channels"] not in ("two", "three"):
        raise ValueError("RGB model inputs must be two selected channels or all three RGB channels.")
    if any(int(s[k]) not in (0, 1, 2, 3) for k in ("cellpose_primary", "cellpose_secondary")):
        raise ValueError("Cellpose RGB channel choices must be 0, 1, 2, or 3.")
    if (s["image_format"] == "rgb" and s["rgb_model_channels"] == "two"
            and int(s["cellpose_primary"]) == 0 and int(s["cellpose_secondary"]) != 0):
        raise ValueError("With Cellpose grayscale as channel 1, channel 2 must be None (0). Select an R/G/B primary channel if your model uses a second signal channel.")
    s["input_dir"] = str(Path(s["input_dir"]).expanduser().resolve())
    s["output_dir"] = str(Path(s["output_dir"]).expanduser().resolve())
    if not Path(s["input_dir"]).is_dir():
        raise ValueError("Select an existing input folder.")
    if inside(s["input_dir"], s["output_dir"]):
        raise ValueError("The output folder must not be the input folder or one of its parents.")
    if Path(s["output_dir"]).exists() and not Path(s["output_dir"]).is_dir():
        raise ValueError("The output path is an existing file. Choose a folder.")
    positive = ("pixel_size", "scale", "diameter", "batch_size", "csa_max",
                "csa_step", "feret_max", "feret_step", "preview_size")
    for key in positive:
        if not math.isfinite(float(s[key])) or float(s[key]) <= 0:
            raise ValueError(f"{key.replace('_', ' ')} must be positive and finite.")
    for key in ("channel", "time", "z", "min_size", "area_min", "area_max"):
        if not math.isfinite(float(s[key])) or float(s[key]) < 0:
            raise ValueError(f"{key.replace('_', ' ')} must be nonnegative and finite.")
    if s["area_max"] > 0 and s["area_max"] <= s["area_min"]:
        raise ValueError("The maximum CSA filter must exceed the minimum (or be 0 to disable it).")
    if not math.isfinite(s["flow_threshold"]) or s["flow_threshold"] < 0:
        raise ValueError("Flow threshold must be finite and nonnegative.")
    if not math.isfinite(s["cellprob_threshold"]):
        raise ValueError("Cell probability threshold must be finite.")
    for prefix in ("csa", "feret"):
        ratio = s[f"{prefix}_max"] / s[f"{prefix}_step"]
        if ratio > 500 or not np.isclose(ratio, round(ratio)):
            raise ValueError(f"{prefix.upper()} distribution maximum must be a multiple of its bin width (at most 500 bins).")
    if require_model and s["mode"] == "full":
        if not s.get("model") or not Path(s["model"]).expanduser().is_file():
            raise ValueError("Select your custom Cellpose 3 model file.")
        s["model"] = str(Path(s["model"]).expanduser().resolve())
    return s


def discover_files(settings):
    root, out = Path(settings["input_dir"]), Path(settings["output_dir"])
    candidates = root.rglob("*") if settings["recursive"] else root.iterdir()
    extensions = (".tif", ".tiff") if settings["mode"] == "metrics" else (".tif", ".tiff", ".czi")
    return sorted([p for p in candidates if p.is_file() and p.suffix.lower() in extensions
                   and not inside(p, out)
                   and (settings["mode"] == "metrics" or not p.stem.lower().endswith("_masks"))],
                  key=lambda p: str(p.relative_to(root)).lower())


def scene_numbers(path, settings):
    if settings["mode"] == "metrics" or not settings["all_scenes"]:
        return [0]
    if path.suffix.lower() == ".czi":
        from aicsimageio import AICSImage
        return list(range(len(AICSImage(str(path)).scenes)))
    with tiff.TiffFile(path) as image:
        return list(range(len(image.series)))


def image_id(path, scene, settings):
    relative = str(Path(path).relative_to(settings["input_dir"]))
    stem = re.sub(r"(?i)(\.ome)?\.(tiff?|czi)$", "", Path(path).name)
    stem = re.sub(r"[^\w.\-]+", "_", stem)[:90]
    return f"{stem}__{hashlib.sha1(relative.encode('utf-8')).hexdigest()[:8]}_S{scene + 1:03d}"


def ome_spacing(xml, scene):
    if not xml:
        return None
    try:
        images = [node for node in ET.fromstring(xml) if node.tag.endswith("}Image")]
        pixels = next(node for node in images[scene] if node.tag.endswith("}Pixels"))
        multipliers = {"µm": 1., "μm": 1., "um": 1., "nm": .001,
                       "mm": 1000., "m": 1e6}
        result = []
        for axis in ("Y", "X"):
            unit = pixels.attrib.get(f"PhysicalSize{axis}Unit", "µm")
            if unit not in multipliers:
                return None
            result.append(float(pixels.attrib[f"PhysicalSize{axis}"]) * multipliers[unit])
        return tuple(result) if all(np.isfinite(result)) and min(result) > 0 else None
    except (ET.ParseError, KeyError, IndexError, StopIteration, ValueError):
        return None


def select_tiff_plane(array, axes, settings):
    """Use TIFF axes; preserve RGB when requested instead of selecting a component."""
    axes = axes.upper()
    if settings["axes"] != "auto" and array.ndim != 2:
        if array.ndim != 3:
            raise ValueError("The manual TIFF layout applies to 3D arrays only. Use Auto for OME multidimensional files.")
        axes = settings["axes"]
    if len(axes) != array.ndim or len(set(axes)) != len(axes):
        raise ValueError(f"Unsupported TIFF axes {axes} for shape {array.shape}.")
    if settings["image_format"] == "rgb":
        color_axes = [axis for axis in "SC" if axis in axes and array.shape[axes.index(axis)] >= 3]
        if len(color_axes) != 1:
            raise ValueError(f"RGB mode requires exactly one RGB sample/channel axis of size >=3; found {axes}, {array.shape}. Use Single-channel for fluorescence stacks.")
        color_axis = color_axes[0]
        selectors, remaining = [], []
        for axis, size in zip(axes, array.shape):
            if axis in "YX":
                selectors.append(slice(None))
                remaining.append(axis)
            elif axis == color_axis:
                selectors.append(slice(0, 3))
                remaining.append("S")
            else:
                index = {"T": settings["time"], "Z": settings["z"]}.get(axis)
                if index is None:
                    if size != 1:
                        raise ValueError(f"Ambiguous TIFF axis '{axis}' for RGB input of shape {array.shape}.")
                    index = 0
                if not 0 <= index < size:
                    raise ValueError(f"Axis {axis}: requested index {index}, only {size} position(s) available.")
                selectors.append(index)
        if sorted(remaining) != ["S", "X", "Y"]:
            raise ValueError(f"Expected two spatial axes and one RGB axis; found {remaining}.")
        plane = array[tuple(selectors)].transpose(tuple(remaining.index(a) for a in "YXS"))
        return plane, {"axes": axes, "channels": array.shape[axes.index(color_axis)],
                       "channel_used": "RGB"}
    # RGB samples are a channel axis; do not confuse them with an OME scene.
    if "S" in axes and "C" not in axes:
        axes = axes.replace("S", "C")
    channel_count = array.shape[axes.index("C")] if "C" in axes else 1
    selected_channel = settings["channel"] if channel_count > 1 else 0
    selectors, remaining = [], []
    for axis, size in zip(axes, array.shape):
        if axis in "YX":
            selectors.append(slice(None))
            remaining.append(axis)
            continue
        index = {"C": selected_channel, "T": settings["time"], "Z": settings["z"]}.get(axis)
        if index is None:
            if size != 1:
                raise ValueError(f"Ambiguous TIFF axis '{axis}' in shape {array.shape}. Select Channels first or Channels last if this is a channel stack.")
            index = 0
        if not 0 <= index < size:
            raise ValueError(f"Axis {axis}: requested index {index}, but only {size} position(s) exist (numbering starts at 0).")
        selectors.append(index)
    plane = array[tuple(selectors)]
    if remaining == ["X", "Y"]:
        plane = plane.T
    elif remaining != ["Y", "X"]:
        raise ValueError(f"Expected Y/X spatial axes, found {remaining}.")
    return plane, {"axes": axes, "channels": channel_count, "channel_used": selected_channel}


def load_plane(path, scene, settings):
    metadata_spacing = None
    if path.suffix.lower() == ".czi":
        try:
            from aicsimageio import AICSImage
        except ImportError as exc:
            raise RuntimeError("CZI input requires aicsimageio and aicspylibczi in this Python environment.") from exc
        image = AICSImage(str(path))
        image.set_scene(scene)
        selected = settings["channel"] if image.dims.C > 1 else 0
        if settings["image_format"] == "rgb" and image.dims.C < 3:
            raise ValueError("RGB CZI mode requires three image channels. Use Single-channel if your CZI has one or two fluorescence channels.")
        for axis, index in (("T", settings["time"]), ("Z", settings["z"])):
            size = getattr(image.dims, axis)
            if not 0 <= index < size:
                raise ValueError(f"Axis {axis}: requested index {index}, but only {size} position(s) exist.")
        if settings["image_format"] == "rgb":
            plane = image.get_image_dask_data("YXC", T=settings["time"], Z=settings["z"]).compute()[..., :3]
            channel_used = "RGB"
        else:
            if not 0 <= selected < image.dims.C:
                raise ValueError(f"Channel index {selected} is unavailable (CZI has {image.dims.C} channels).")
            plane = image.get_image_dask_data("YX", C=selected, T=settings["time"], Z=settings["z"]).compute()
            channel_used = selected
        physical = image.physical_pixel_sizes
        if physical.Y and physical.X:
            metadata_spacing = (float(physical.Y), float(physical.X))
        info = {"axes": image.dims.order, "channels": image.dims.C, "channel_used": channel_used}
    else:
        with tiff.TiffFile(path) as image:
            series = image.series[scene]
            plane, info = select_tiff_plane(series.asarray(), series.axes, settings)
            metadata_spacing = ome_spacing(image.ome_metadata, scene)
    plane = np.asarray(plane)
    expected = "YX" if settings["image_format"] == "single" else "YXS (RGB)"
    if (settings["image_format"] == "single" and plane.ndim != 2 or
            settings["image_format"] == "rgb" and (plane.ndim != 3 or plane.shape[-1] != 3) or
            min(plane.shape[:2]) < 2):
        raise ValueError(f"Expected {expected} plane, found shape {plane.shape}.")
    if settings["calibration"] == "metadata" and metadata_spacing is not None:
        spacing, calibration_source = metadata_spacing, "Image metadata"
    else:
        spacing = (settings["pixel_size"], settings["pixel_size"])
        calibration_source = "Manual" if settings["calibration"] == "manual" else "Manual fallback (metadata unavailable)"
    info.update({"original_shape": list(plane.shape[:2]), "spacing_yx_um": list(spacing),
                 "calibration_source": calibration_source})
    return plane, info


def normalize_nonzero_2d(array):
    array = np.nan_to_num(np.asarray(array, dtype=np.float32), nan=0., posinf=0., neginf=0.)
    array = np.maximum(array, 0.)
    result = np.zeros_like(array)
    valid = array > 0
    if valid.any():
        low, high = float(array[valid].min()), float(array[valid].max())
        result[valid] = (array[valid] - low) / (high - low) if high > low else 1.
    return result


def prepare_plane(plane, info, settings):
    original_shape = plane.shape[:2]
    target_shape = tuple(max(2, int(round(n * settings["scale"]))) for n in original_shape)
    if settings["image_format"] == "rgb":
        if plane.ndim != 3 or plane.shape[2] != 3:
            raise ValueError(f"Expected RGB plane (Y, X, 3), found {plane.shape}.")
        # Preserve the exact RGB input used for Cellpose GUI training. Do NOT
        # normalize each color channel independently or apply fluorescence
        # nonzero normalization; both would alter the model's input colors.
        if target_shape != original_shape:
            if plane.dtype == np.uint8:
                method = Image.Resampling.LANCZOS if settings["scale"] < 1 else Image.Resampling.BICUBIC
                plane = np.asarray(Image.fromarray(plane).resize((target_shape[1], target_shape[0]), resample=method))
            else:
                source_dtype = plane.dtype
                plane = resize(plane.astype(np.float32), (*target_shape, 3), order=3,
                               preserve_range=True, anti_aliasing=settings["scale"] < 1,
                               mode="reflect").astype(source_dtype)
        info = dict(info)
        info["spacing_yx_um"] = [float(p * old / new) for p, old, new in
                                 zip(info["spacing_yx_um"], original_shape, target_shape)]
        info["processed_shape"] = list(target_shape)
        return plane, info
    plane = np.nan_to_num(np.asarray(plane, dtype=np.float32), nan=0., posinf=0., neginf=0.)
    plane = np.maximum(plane, 0.)
    if target_shape != original_shape:
        plane = resize(plane, target_shape, order=3, preserve_range=True,
                       anti_aliasing=settings["scale"] < 1, mode="reflect").astype(np.float32)
        plane = np.maximum(plane, 0.)
    if settings["normalize_nonzero"]:
        plane = normalize_nonzero_2d(plane)
    info = dict(info)
    info["spacing_yx_um"] = [float(p * old / new) for p, old, new in
                             zip(info["spacing_yx_um"], original_shape, target_shape)]
    info["processed_shape"] = list(plane.shape)
    return plane, info


def has_image_contrast(plane):
    """Check spatial contrast, not the difference between constant RGB channels."""
    if not np.any(plane):
        return False
    if plane.ndim == 3:
        return any(np.ptp(plane[..., c]) > 0 for c in range(plane.shape[-1]))
    return bool(np.ptp(plane) > 0)


def validate_mask(mask):
    mask = np.asarray(mask)
    if mask.ndim != 2 or mask.dtype.kind not in "bui":
        raise ValueError(f"Label masks must be 2D integer TIFFs; found shape={mask.shape}, dtype={mask.dtype}.")
    if mask.size and (mask.min() < 0 or mask.max() > np.iinfo(np.uint32).max):
        raise ValueError("Labels must be nonnegative 32-bit integers (0 is background).")
    return mask.astype(np.uint32, copy=False)


def make_cellpose_model(settings, emit):
    cp_version = installed_version("cellpose")
    if cp_version == "not installed" or cp_version.split(".")[0] != "3":
        raise RuntimeError(f"MyoSeg requires Cellpose 3.x; detected: {cp_version}. Use your Cellpose 3 environment, not a Cellpose-SAM / v4 environment.")
    import torch
    from cellpose import models
    cuda = torch.cuda.is_available()
    if settings["device"] == "cuda" and not cuda:
        raise RuntimeError("CUDA GPU was requested but PyTorch cannot access CUDA. Select Auto or CPU.")
    use_gpu = cuda and settings["device"] != "cpu"
    emit({"kind": "log", "text": f"Loading Cellpose {cp_version} on {'CUDA GPU' if use_gpu else 'CPU'}."})
    nchan = 3 if settings["image_format"] == "rgb" and settings["rgb_model_channels"] == "three" else 2
    emit({"kind": "log", "text": f"Cellpose model input: {nchan} channels. Match the training configuration."})
    model = models.CellposeModel(gpu=use_gpu, pretrained_model=settings["model"],
                                model_type=None, nchan=nchan)
    return model


def segment_plane(model, plane, settings):
    if not has_image_contrast(plane):
        return np.zeros(plane.shape[:2], dtype=np.uint32)
    rgb = settings["image_format"] == "rgb"
    if not rgb:
        channels = [0, 0]
    elif settings["rgb_model_channels"] == "three":
        channels = None  # Cellpose 3: preserve all RGB channels for nchan=3 models.
    else:
        channels = [int(settings["cellpose_primary"]), int(settings["cellpose_secondary"])]
    result = model.eval(plane, channels=channels, channel_axis=-1 if rgb else None, do_3D=False,
                        diameter=settings["diameter"], flow_threshold=settings["flow_threshold"],
                        cellprob_threshold=settings["cellprob_threshold"], min_size=settings["min_size"],
                        batch_size=settings["batch_size"], normalize=settings["cellpose_normalize"],
                        invert=settings["invert"], resample=True)
    mask = validate_mask(result[0])
    if mask.shape != plane.shape[:2]:
        raise ValueError(f"Cellpose returned mask shape {mask.shape}, expected {plane.shape[:2]}.")
    return mask


def minimum_feret(region, spacing):
    """Minimum hull width of pixel centres, as in the source notebook.

    Using boundary pixel centres gives the same hull as all region pixels.
    Degenerate (point/line) regions have a minimum width of zero.
    """
    boundary = region.image & ~binary_erosion(region.image, border_value=0)
    points = np.argwhere(boundary).astype(float) * np.asarray(spacing)
    if len(points) < 3:
        return 0.
    try:
        hull = ConvexHull(points)
    except QhullError:
        return 0.
    vertices = points[hull.vertices]
    # Translation improves numerical conditioning without changing widths.
    vertices -= vertices.mean(axis=0)
    edges = np.roll(vertices, -1, axis=0) - vertices
    lengths = np.linalg.norm(edges, axis=1)
    normals = np.column_stack((-edges[:, 1], edges[:, 0])) / lengths[:, None]
    widths = [np.ptp(vertices @ normal) for normal in normals]
    return float(min(widths))


def distribution(values, maximum, step, unit):
    edges = np.linspace(0, maximum, int(round(maximum / step)) + 1)
    names = [f"{a:g}-{b:g} {unit}" for a, b in zip(edges[:-1], edges[1:])]
    names.append(f">={maximum:g} {unit}")
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    indices = np.searchsorted(edges, values, side="right") - 1
    counts = np.bincount(indices, minlength=len(names)) if len(values) else np.zeros(len(names), dtype=int)
    count_map = {key: int(value) for key, value in zip(names, counts)}
    pct_map = {key: float(value / len(values) * 100) if len(values) else 0.
               for key, value in zip(names, counts)}
    return count_map, pct_map


def measure_mask(mask, info, settings, identifier, filename, scene):
    mask = validate_mask(mask)
    spacing = info["spacing_yx_um"]
    rows = []
    for region in regionprops(mask):
        area = float(region.area * spacing[0] * spacing[1])
        border = (region.bbox[0] == 0 or region.bbox[1] == 0 or
                  region.bbox[2] == mask.shape[0] or region.bbox[3] == mask.shape[1])
        reasons = []
        if settings["exclude_border"] and border:
            reasons.append("Image border")
        if area < settings["area_min"]:
            reasons.append("Below minimum CSA")
        if settings["area_max"] > 0 and area > settings["area_max"]:
            reasons.append("Above maximum CSA")
        feret = minimum_feret(region, spacing) if settings["compute_feret"] else np.nan
        rows.append([identifier, filename, scene + 1, int(region.label), area, feret,
                     border, not reasons, "; ".join(reasons)])
    detail = pd.DataFrame(rows, columns=DETAIL_COLUMNS)
    included = detail.loc[detail["Included"].astype(bool)]
    areas = included["Area_µm²"].to_numpy(dtype=float)
    ferets = included["Feret_Diameter_µm"].to_numpy(dtype=float)
    def statistic(array, function):
        finite = array[np.isfinite(array)]
        return float(function(finite)) if len(finite) else None
    summary = {"Image_ID": identifier, "Filename": filename, "Scene": scene + 1,
               "Number_of_Labels": len(included), "Detected_Labels": len(detail),
               "Excluded_Labels": len(detail) - len(included),
               "Mean_CSA_µm²": statistic(areas, np.mean),
               "Median_CSA_µm²": statistic(areas, np.median),
               "Max_CSA_µm²": statistic(areas, np.max),
               "Min_CSA_µm²": statistic(areas, np.min),
               "Mean_Feret_Diameter_µm": statistic(ferets, np.mean),
               "Pixel_size_Y_µm": spacing[0], "Pixel_size_X_µm": spacing[1],
               "Calibration": info["calibration_source"],
               "Status": "No fibers detected" if not len(detail) else
                         ("All fibers excluded" if not len(included) else "OK")}
    csa_counts, csa_pct = distribution(areas, settings["csa_max"], settings["csa_step"], "µm²")
    feret_counts, feret_pct = distribution(ferets, settings["feret_max"], settings["feret_step"], "µm")
    identity = {"Image_ID": identifier, "Filename": filename, "Scene": scene + 1}
    return detail, {"summary": summary, "csa_counts": {**identity, **csa_counts},
                    "csa_pct": {**identity, **csa_pct}, "feret_counts": {**identity, **feret_counts},
                    "feret_pct": {**identity, **feret_pct}}


def preview_png(plane, mask=None, max_size=1400):
    """Small preview with true RGB color for brightfield and grayscale for IF."""
    height, width = plane.shape[:2]
    scale = min(1., max_size / max(height, width))
    shape = (max(1, int(round(height * scale))), max(1, int(round(width * scale))))
    # Reduce before making temporary floating-point/RGB arrays for large images.
    factor = max(1, max(height, width) // max_size)
    if plane.ndim == 3 and plane.shape[-1] == 3:
        sampled = np.asarray(plane[::factor, ::factor])
        if sampled.dtype == np.uint8:
            rgb = sampled
        else:
            sampled = np.asarray(sampled, dtype=np.float32)
            if np.issubdtype(plane.dtype, np.integer):
                sampled *= 255. / np.iinfo(plane.dtype).max
            elif sampled.min() >= 0 and sampled.max() <= 1.:
                sampled *= 255.
            else:
                sampled *= 255. / max(1., float(np.nanpercentile(sampled, 99.9)))
            rgb = np.clip(np.nan_to_num(sampled), 0, 255).astype(np.uint8)
        if rgb.shape[:2] != shape:
            rgb = np.asarray(Image.fromarray(rgb).resize((shape[1], shape[0]),
                                        resample=Image.Resampling.BILINEAR)).copy()
        else:
            rgb = rgb.copy()
    else:
        sampled = np.asarray(plane[::factor, ::factor], dtype=np.float32)
        base = resize(sampled, shape, order=1, preserve_range=True,
                      anti_aliasing=True) if shape != sampled.shape else sampled
        finite = base[np.isfinite(base)]
        lo, hi = np.percentile(finite, (1, 99)) if finite.size else (0., 1.)
        grey = np.clip((np.nan_to_num(base) - lo) / (hi - lo), 0, 1) if hi > lo else np.zeros(shape)
        rgb = np.repeat((grey * 255).astype(np.uint8)[..., None], 3, axis=2)
    if mask is not None:
        # Preview boundaries are computed on reduced labels; measurements use the full mask.
        small_mask = mask[::factor, ::factor]
        if shape != small_mask.shape:
            small_mask = resize(small_mask, shape, order=0, preserve_range=True,
                                anti_aliasing=False).astype(np.uint32)
        boundaries = find_boundaries(small_mask, mode="inner")
        rgb[boundaries] = (0, 255, 255) if plane.ndim == 3 else (191, 135, 255)
    stream = io.BytesIO()
    Image.fromarray(rgb).save(stream, format="PNG")
    return stream.getvalue()


def save_workbook(root, records, settings):
    """Rebuild all result sheets from this run's valid per-image records."""
    root = Path(root)
    tables = root / "Tables"
    tables.mkdir(parents=True, exist_ok=True)
    sheets = {
        "Summary": pd.DataFrame([r["metrics"]["summary"] for r in records]),
        "CSA_Distribution": pd.DataFrame([r["metrics"]["csa_counts"] for r in records]),
        "Feret_Distribution": pd.DataFrame([r["metrics"]["feret_counts"] for r in records]),
        "CSA_Distribution_%": pd.DataFrame([r["metrics"]["csa_pct"] for r in records]),
        "Feret_Distribution_%": pd.DataFrame([r["metrics"]["feret_pct"] for r in records]),
    }
    for name, frame in sheets.items():
        atomic_csv(tables / f"{name}.csv", frame)
    target = root / "segmentation_results.xlsx"
    if target.exists():
        backup = root / "Log" / "Backups" / f"segmentation_results_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.xlsx"
        backup.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(target, backup)
    temp = root / f".results_{uuid.uuid4().hex}.xlsx"
    try:
        with pd.ExcelWriter(temp, engine="openpyxl") as writer:
            for name, frame in sheets.items():
                frame.to_excel(writer, sheet_name=name, index=False)
            # Write detailed labels incrementally; split at Excel's row limit.
            sheet_index, offset = 1, 0
            for record in records:
                detail = pd.read_csv(root / record["detail_path"])
                cursor = 0
                while cursor < len(detail) or (not len(detail) and offset == 0):
                    capacity = 1_048_575 - offset
                    part = detail.iloc[cursor:cursor + capacity]
                    sheet_name = "Detailed_Labels" if sheet_index == 1 else f"Detailed_Labels_{sheet_index}"
                    part.to_excel(writer, sheet_name=sheet_name, index=False,
                                  startrow=offset + (1 if offset else 0), header=offset == 0)
                    offset += len(part)
                    cursor += len(part)
                    if not len(detail):
                        break
                    if offset == 1_048_575:
                        sheet_index += 1
                        offset = 0
            pd.DataFrame([{"Parameter": k, "Value": str(v)} for k, v in settings.items()]).to_excel(
                writer, sheet_name="Run_Parameters", index=False)
        os.replace(temp, target)
    finally:
        temp.unlink(missing_ok=True)
    return str(target)


def run_pipeline(settings, emit=None, stop_event=None, first_only=False, model_factory=None):
    """Process one image at a time. Checkpoints are committed after each stage.

    Re-running with unchanged settings reuses valid files; changes to image,
    model, calibration or segmentation parameters invalidate the affected stage.
    Measurement-only changes never trigger segmentation again.
    """
    emit = emit or (lambda event: None)
    stop_event = stop_event or threading.Event()
    s = validate_settings(settings)
    files = discover_files(s)
    if not files:
        raise ValueError("No compatible input files were found in the selected folder.")
    root = Path(s["output_dir"])
    for name in ("Prepared_images", "Masks", "Previews", "Tables/Per_image", "Log"):
        (root / name).mkdir(parents=True, exist_ok=True)
    manifest_path = root / "Log" / "checkpoints.json"
    if manifest_path.exists():
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            if manifest.get("schema") != 1:
                raise ValueError("Unsupported checkpoint schema.")
        except (ValueError, OSError) as exc:
            raise RuntimeError("Cannot read existing checkpoints. Choose a new output folder or restore Log/checkpoints.json.") from exc
    else:
        manifest = {"schema": 1, "application": "MyoSeg", "version": APP_VERSION, "images": {}}
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    log_path = root / "Log" / f"run_{run_id}.log"
    atomic_json(root / "Log" / f"settings_{run_id}.json", {**s, "version": APP_VERSION, "started_utc": utc_now()})
    def log(text):
        with log_path.open("a", encoding="utf-8") as stream:
            stream.write(f"{utc_now()}  {text}\n")
        emit({"kind": "log", "text": text})
    errors, tasks = [], []
    for path in files:
        if stop_event.is_set():
            break
        try:
            for scene in scene_numbers(path, s):
                tasks.append((path, scene))
                if first_only:
                    break
        except Exception as exc:
            errors.append({"Filename": str(path), "Scene": None, "Error": str(exc)})
            log(f"Could not inspect {path.name}: {exc}")
        if first_only and tasks:
            break
    if not tasks:
        atomic_csv(root / "Tables" / "Errors.csv", pd.DataFrame(errors, columns=["Filename", "Scene", "Error"]))
        raise ValueError("No readable image scenes were found. See the run log for details.")
    log(f"Found {len(tasks)} image(s)/scene(s). Output: {root}")
    factory = model_factory or make_cellpose_model
    model, valid_records, done, reused = None, [], 0, 0
    prep_keys = ("image_format", "channel", "time", "z", "axes", "calibration", "pixel_size", "scale", "normalize_nonzero")
    seg_keys = ("rgb_model_channels", "cellpose_primary", "cellpose_secondary", "diameter", "flow_threshold", "cellprob_threshold", "min_size", "batch_size", "cellpose_normalize", "invert")
    measure_keys = ("exclude_border", "area_min", "area_max", "compute_feret", "csa_max", "csa_step", "feret_max", "feret_step")
    model_stamp = file_stamp(s["model"]) if s["mode"] == "full" else None
    for index, (path, scene) in enumerate(tasks):
        if stop_event.is_set():
            log("Stop requested. Keeping completed images and saved checkpoints.")
            break
        identifier = image_id(path, scene, s)
        old = manifest["images"].get(identifier, {})
        record = dict(old)
        filename = str(path.relative_to(s["input_dir"]))
        emit({"kind": "progress", "current": index, "total": len(tasks),
              "text": f"{filename} · scene {scene + 1}"})
        emit({"kind":"file","file":filename+f" · scene {scene+1}","state":"processing"})
        outcome = "completed"
        log(f"Processing {filename} · scene {scene + 1}")
        try:
            image, mask, info = None, None, None
            prep_sig = signature({"source": file_stamp(path), "scene": scene,
                                  "parameters": {k: s[k] for k in prep_keys}, "version": APP_VERSION})
            prepared_path = root / "Prepared_images" / f"{identifier}.ome.tif"
            mask_path = root / "Masks" / f"{identifier}_masks.tif"
            if s["mode"] == "metrics":
                # Existing masks are measured at their own resolution; no resize or normalization.
                with tiff.TiffFile(path) as image_file:
                    mask = validate_mask(image_file.asarray())
                    physical = ome_spacing(image_file.ome_metadata, 0) if s["calibration"] == "metadata" else None
                spacing = physical or (s["pixel_size"], s["pixel_size"])
                info = {"spacing_yx_um": list(spacing), "processed_shape": list(mask.shape),
                        "calibration_source": "Image metadata" if physical else
                        ("Manual" if s["calibration"] == "manual" else "Manual fallback (metadata unavailable)")}
                mask_path = path
            else:
                prep_valid = (not s["force"] and old.get("prep_sig") == prep_sig
                              and prepared_path.exists() and old.get("info") is not None
                              and old.get("prepared_stamp") == file_stamp(prepared_path))
                if prep_valid:
                    info = old["info"]
                else:
                    image, info = load_plane(path, scene, s)
                    image, info = prepare_plane(image, info, s)
                    # Keep RGB uint8 pixel values (including stain color) intact.
                    write_image(prepared_path, image, info["spacing_yx_um"])
                    record.update({"prep_sig": prep_sig, "prepared_stamp": file_stamp(prepared_path),
                                   "info": info, "source": str(path), "scene": scene})
                    # A new preparation invalidates all later stages, even if a write failed previously.
                    record.pop("seg_sig", None)
                    record.pop("metric_sig", None)
                    manifest["images"][identifier] = record
                    atomic_json(manifest_path, manifest)
                if s["mode"] == "full":
                    seg_sig = signature({"prepared": file_stamp(prepared_path), "model": model_stamp,
                                         "parameters": {k: s[k] for k in seg_keys},
                                         "cellpose": installed_version("cellpose"), "version": APP_VERSION})
                    seg_valid = (not s["force"] and record.get("seg_sig") == seg_sig and mask_path.exists()
                                 and record.get("mask_stamp") == file_stamp(mask_path))
                    if not seg_valid:
                        if image is None:
                            image = tiff.imread(prepared_path)
                        if has_image_contrast(image):
                            if model is None:
                                model = factory(s, lambda event: log(event["text"]))
                            mask = segment_plane(model, image, s)
                        else:
                            mask = np.zeros(image.shape[:2], dtype=np.uint32)
                            log("No intensity contrast: saving an empty mask.")
                        write_image(mask_path, mask, info["spacing_yx_um"])
                        record.update({"seg_sig": seg_sig, "mask_stamp": file_stamp(mask_path)})
                        record.pop("metric_sig", None)
                        manifest["images"][identifier] = record
                        atomic_json(manifest_path, manifest)
                    else:
                        reused += 1
                        log("Reusing a valid segmentation checkpoint.")
            if s["mode"] != "prepare":
                metric_sig = signature({"mask": file_stamp(mask_path), "spacing": info["spacing_yx_um"],
                                        "parameters": {k: s[k] for k in measure_keys}, "version": APP_VERSION})
                detail_path = root / "Tables" / "Per_image" / f"{identifier}.csv"
                metric_valid = (not s["force"] and record.get("metric_sig") == metric_sig
                                and detail_path.exists() and "metrics" in record
                                and record.get("detail_stamp") == file_stamp(detail_path))
                if not metric_valid:
                    if mask is None:
                        mask = validate_mask(tiff.imread(mask_path))
                    detail, metrics = measure_mask(mask, info, s, identifier, filename, scene)
                    atomic_csv(detail_path, detail)
                    record.update({"metric_sig": metric_sig, "metrics": metrics,
                                   "detail_path": str(detail_path.relative_to(root)),
                                   "detail_stamp": file_stamp(detail_path), "updated_utc": utc_now()})
                    manifest["images"][identifier] = record
                    atomic_json(manifest_path, manifest)
                valid_records.append(record)
                summary = record["metrics"]["summary"]
                log(f"{summary['Number_of_Labels']} included / {summary['Detected_Labels']} detected fibers. {summary['Status']}.")
            preview_path = root / "Previews" / f"{identifier}.png"
            preview_sig = signature({"image": file_stamp(prepared_path) if s["mode"] != "metrics" else file_stamp(mask_path),
                                     "mask": file_stamp(mask_path) if s["mode"] != "prepare" else None,
                                     "size": s["preview_size"]})
            show_preview = first_only or index == len(tasks) - 1 or index == 0
            if s["save_previews"] or show_preview:
                if record.get("preview_sig") == preview_sig and preview_path.exists():
                    png = preview_path.read_bytes()
                else:
                    if s["mode"] == "metrics":
                        if mask is None:
                            mask = validate_mask(tiff.imread(mask_path))
                        image = (mask > 0).astype(np.float32)
                    elif image is None:
                        image = tiff.imread(prepared_path)
                    if s["mode"] == "full" and mask is None:
                        mask = validate_mask(tiff.imread(mask_path))
                    png = preview_png(image, mask, s["preview_size"])
                    if s["save_previews"]:
                        preview_path.write_bytes(png)
                        record["preview_sig"] = preview_sig
                if show_preview:
                    emit({"kind": "preview", "png": png,
                          "text": f"{filename} · scene {scene + 1} · {info['processed_shape'][1]} × {info['processed_shape'][0]} px · {info['spacing_yx_um'][1]:.5g} µm/px (X), {info['spacing_yx_um'][0]:.5g} µm/px (Y). Preview only; masks retain full resolution."})
            manifest["images"][identifier] = record
            atomic_json(manifest_path, manifest)
            done += 1
        except Exception as exc:
            outcome = "failed"
            errors.append({"Filename": filename, "Scene": scene + 1, "Error": str(exc)})
            log(f"ERROR: {filename}, scene {scene + 1}: {exc}")
            with log_path.open("a", encoding="utf-8") as stream:
                stream.write(traceback.format_exc() + "\n")
        finally:
            emit({"kind":"file","file":filename+f" · scene {scene+1}","state":outcome})
            emit({"kind": "progress", "current": index + 1, "total": len(tasks),
                  "text": f"Completed {index + 1} / {len(tasks)} image(s)/scene(s)"})
    # A one-image test retains earlier rows whose results are still valid.
    # A stopped full run also retains every valid checkpoint from this input set.
    if s["mode"] != "prepare" and (first_only or stop_event.is_set()):
        known = {r["metrics"]["summary"]["Image_ID"] for r in valid_records}
        current_ids = {image_id(path, scene, s) for path, scene in tasks}
        for identifier, record in manifest["images"].items():
            if identifier in known or (not first_only and identifier not in current_ids):
                continue
            # Only retain prior results with the same input root/settings and a still-valid mask/detail.
            if record.get("result_context") != result_context(s, model_stamp):
                continue
            source = Path(record.get("source", ""))
            if not source.is_file() or record.get("source_stamp") != file_stamp(source):
                continue
            mask_file = Path(record.get("mask_path", ""))
            detail_file = root / record.get("detail_path", "missing")
            if (mask_file.is_file() and detail_file.is_file() and record.get("metrics")
                    and record.get("result_mask_stamp") == file_stamp(mask_file)
                    and record.get("detail_stamp") == file_stamp(detail_file)):
                valid_records.append(record)
    # Store the context needed to preserve earlier valid rows after a one-image test/stop.
    context = result_context(s, model_stamp)
    for record in valid_records:
        identifier = record["metrics"]["summary"]["Image_ID"]
        path = Path(s["input_dir"]) / record["metrics"]["summary"]["Filename"]
        if identifier in {image_id(p, scene, s) for p, scene in tasks}:
            record.update({"result_context": context, "source": str(path), "source_stamp": file_stamp(path),
                           "mask_path": str(path if s["mode"] == "metrics" else root / "Masks" / f"{identifier}_masks.tif")})
            record["result_mask_stamp"] = file_stamp(record["mask_path"])
            manifest["images"][identifier] = record
    atomic_json(manifest_path, manifest)
    atomic_csv(root / "Tables" / "Errors.csv", pd.DataFrame(errors, columns=["Filename", "Scene", "Error"]))
    workbook = None
    if valid_records:
        valid_records.sort(key=lambda r: (r["metrics"]["summary"]["Filename"], r["metrics"]["summary"]["Scene"]))
        workbook = save_workbook(root, valid_records, s)
        log(f"Results exported to {workbook}")
    elif s["mode"] != "prepare":
        log("No valid measurements to export. Any existing workbook has been kept.")
    log(f"Finished: {done} successful, {len(errors)} error(s), {reused} reused mask(s).")
    return {"successful": done, "errors": len(errors), "reused": reused,
            "stopped": stop_event.is_set(), "workbook": workbook, "output_dir": str(root),
            "log_path": str(log_path), "records": len(valid_records)}


def result_context(settings, model_stamp):
    ignored = {"force", "save_previews", "preview_size", "device", "recursive", "all_scenes"}
    return signature({"settings": {k: v for k, v in settings.items() if k not in ignored},
                      "model": model_stamp, "version": APP_VERSION})


def preview_input(settings):
    s = validate_settings(settings, require_model=False)
    files = discover_files(s)
    if not files:
        raise ValueError("No compatible input files found.")
    path = files[0]
    if s["mode"] == "metrics":
        with tiff.TiffFile(path) as source:
            mask = validate_mask(source.asarray())
        return {"kind": "preview", "png": preview_png((mask > 0).astype(np.float32), mask, s["preview_size"]),
                "text": f"{path.name}: {mask.shape[1]} × {mask.shape[0]} px, {len(np.unique(mask[mask > 0]))} distinct labels. Existing masks will not be resized."}
    plane, info = load_plane(path, 0, s)
    prepared, info = prepare_plane(plane, info, s)
    return {"kind": "preview", "png": preview_png(prepared, max_size=s["preview_size"]),
            "text": f"{path.name} · {info['axes']} · channel {info['channel_used']} / {info['channels']} channel(s) · {info['original_shape'][1]} × {info['original_shape'][0]} → {info['processed_shape'][1]} × {info['processed_shape'][0]} px · {info['spacing_yx_um'][1]:.5g} µm/px (X), {info['spacing_yx_um'][0]:.5g} µm/px (Y) · {info['calibration_source']}"}


In [ ]:
from IPython.display import HTML, Markdown, display

display(HTML(r'''
<style>
:root { color-scheme: light; }
body { background: #f3f4f5 !important; }
#rendered_cells {
  position: relative !important;
  inset: auto !important;
  width: min(100%, 1150px) !important;
  max-width: 1150px !important;
  margin: 0 auto !important;
  padding: 34px 20px 68px !important;
  display: block !important;
  text-align: left !important;
  box-sizing: border-box !important;
}
.myoseg-app { width: 100%; min-width: 0; gap: 18px; color: #252525; font-family: Inter, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; }
.myoseg-app * { box-sizing: border-box; }
.ms-hero {
  position: relative; overflow: hidden; padding: 34px 42px 30px;
  border-radius: 20px; color: #fff;
  background: radial-gradient(circle at 88% 10%, #333333 0, transparent 31%), linear-gradient(125deg, #080808 0%, #151515 67%, #252525 100%);
  box-shadow: 0 18px 40px rgba(0, 0, 0, .16);
}
.ms-hero::after { content: ""; position: absolute; right: -100px; top: 35px; width: 290px; height: 290px; border: 1px solid rgba(255,255,255,.14); border-radius: 50%; pointer-events: none; }
.ms-hero-top { display: flex; align-items: center; justify-content: space-between; gap: 16px; }
.ms-mark { width: 43px; height: 43px; display: inline-flex; align-items: center; justify-content: center; background: #e5e5e5; color: #111111; border-radius: 12px; font-size: 15px; font-weight: 800; letter-spacing: -.03em; }
.ms-version { color: #c7c7c7; font-size: 11px; font-weight: 700; letter-spacing: .15em; }
.ms-hero h1 { margin: 22px 0 7px; color: #fff; font-size: clamp(34px, 5vw, 52px); line-height: 1.05; font-weight: 760; letter-spacing: -.045em; }
.ms-hero p { margin: 0; max-width: 620px; color: #d8d8d8; font-size: 15px; line-height: 1.6; }
.ms-hero-foot { display: flex; flex-wrap: wrap; gap: 11px; margin-top: 26px; }
.ms-hero-foot span { padding: 7px 12px; border-radius: 999px; background: rgba(255,255,255,.1); border: 1px solid rgba(255,255,255,.12); color: #ededed; font-size: 11px; font-weight: 650; }
.ms-hero-foot .ms-paper-link { display: inline-flex; align-items: center; margin-left: auto; padding: 7px 13px; border-radius: 999px; background: #e5e5e5; color: #161616; font-size: 11px; font-weight: 750; text-decoration: none; }
.ms-hero-foot .ms-paper-link:hover, .ms-hero-foot .ms-paper-link:focus { background: #f5f5f5; color: #161616; text-decoration: underline; }
.ms-card { width: 100%; min-width: 0; padding: 26px 30px 28px; background: #fff; border: 1px solid #dedede; border-radius: 17px; box-shadow: 0 5px 22px rgba(0, 0, 0, .045); gap: 15px; }
.ms-section-heading { display: flex; gap: 15px; align-items: flex-start; padding-bottom: 16px; border-bottom: 1px solid #ededed; }
.ms-number { flex: none; width: 38px; height: 38px; border-radius: 11px; display: inline-flex; align-items: center; justify-content: center; background: #eeeeee; color: #171717; font-size: 12px; font-weight: 780; letter-spacing: .04em; }
.ms-section-heading h2 { margin: 0 0 4px; color: #222222; font-size: 19px; line-height: 1.25; font-weight: 750; letter-spacing: -.02em; }
.ms-section-heading p { margin: 0; color: #6b6b6b; font-size: 13px; line-height: 1.5; }
.ms-field-label { margin: 4px 0 8px; display: flex; flex-direction: column; gap: 3px; }
.ms-field-label strong { font-size: 13px; color: #2d2d2d; }
.ms-field-label span { color: #6b6b6b; font-size: 12px; line-height: 1.45; }
.ms-chooser { min-width: 0; max-width: 100%; }
.ms-chooser select, .ms-chooser input { max-width: 100%; }
.myoseg-app .widget-label { color: #3a3a3a; font-size: 13px; font-weight: 550; }
.myoseg-app input, .myoseg-app select { border-color: #cfcfcf; border-radius: 7px; }
.myoseg-app input:focus, .myoseg-app select:focus { border-color: #262626; outline-color: #262626; }
.ms-buttons { gap: 10px; margin: 2px 0 0; }
.ms-buttons .widget-button, .ms-buttons button { border-radius: 9px; font-size: 13px; font-weight: 700; box-shadow: none; transition: transform .15s ease, box-shadow .15s ease; }
.ms-buttons .widget-button:hover, .ms-buttons button:hover { transform: translateY(-1px); box-shadow: 0 5px 15px rgba(0, 0, 0, .17); }
.ms-primary-button, .ms-primary-button button { background: #171717 !important; border-color: #171717 !important; color: #fff !important; }
.ms-primary-button:hover, .ms-primary-button button:hover { background: #000000 !important; }
.ms-secondary-button, .ms-secondary-button button { background: #f1f1f1 !important; border: 1px solid #d2d2d2 !important; color: #000000 !important; }
.ms-status { display: inline-flex; align-items: center; gap: 9px; padding: 9px 12px; border-radius: 8px; font-size: 12px; font-weight: 650; }
.ms-status-dot { display: inline-block; width: 8px; height: 8px; border-radius: 50%; background: currentColor; }
.ms-ready { background: #f0f0f0; color: #676767; }
.ms-running { background: #eeeeee; color: #171717; }
.ms-complete { background: #e8e8e8; color: #000000; }
.ms-error { background: #fff0eb; color: #ad4c35; }
.ms-output { width: 100%; min-width: 0; padding: 3px 2px; }
.ms-output .widget-progress { width: min(100%, 520px) !important; }
.ms-output .progress-bar { background: #262626 !important; }
.ms-output .jp-OutputArea pre, .ms-output .output_subarea pre { text-align: left !important; white-space: pre-wrap; overflow-wrap: anywhere; font-size: 12px; line-height: 1.55; }
.ms-output h2 { font-size: 15px; color: #343434; }
@media (max-width: 640px) {
  #rendered_cells { padding: 14px 12px 40px !important; }
  .ms-hero { padding: 25px 23px; }
  .ms-card { padding: 20px 18px; }
  .ms-version { font-size: 9px; letter-spacing: .08em; }
  .ms-hero-foot .ms-paper-link { margin-left: 0; }
  .ms-buttons > * { width: 100% !important; }
}

.myoseg-app .widget-html-content { width: 100%; }
.myoseg-app .widget-checkbox { height: auto; min-height: 29px; }
.myoseg-app .widget-checkbox label { white-space: normal; height: auto; line-height: 1.5; }
.ms-card > .widget-box { min-width: 0; }
.ms-note { padding: 10px 13px; background: #f6f6f6; color: #5a5a5a; border-radius: 9px; font-size: 12px; line-height: 1.6; overflow-wrap: anywhere; }
.ms-help, .ms-output-map { color: #646464; font-size: 12px; line-height: 1.9; }
.ms-output-map strong, .ms-help strong { color: #383838; }
.ms-log { padding: 12px 15px; border: 1px solid #e7e7e7; border-radius: 10px; font-family: ui-monospace, SFMono-Regular, monospace; font-size: 11px; line-height: 1.75; color: #656565; overflow-wrap: anywhere; }
.ms-log > div + div { border-top: 1px solid #f4f4f4; padding-top: 3px; margin-top: 3px; }
.ms-preview { border-radius: 10px; object-fit: contain; max-height: 650px; background: #f8f8f8; }
.ms-progress .progress { height: 8px; background: #ededed; border-radius: 8px; }
.ms-progress .progress-bar { background: #262626; }
.ms-footer { padding: 4px 12px 18px; color: #787878; font-size: 11px; line-height: 1.8; }
.myoseg-app .widget-button:disabled { opacity: .45; }
@media (max-width: 640px) {
  .myoseg-app .widget-inline-hbox { flex-wrap: wrap; height: auto; gap: 4px; }
  .myoseg-app .widget-label, .myoseg-app .widget-inline-hbox > label { width: 100% !important; text-align: left !important; margin-left: 0; }
}
.myoseg-app input[type="checkbox"] { accent-color: #202020; }
.ms-files{max-height:260px;overflow:auto;border:1px solid #e3e4eb;border-radius:12px;background:white}.ms-file{padding:7px 12px;display:flex;justify-content:space-between;gap:12px;border-bottom:1px solid #eef0f2;font-size:12px}.ms-file span{overflow:hidden;text-overflow:ellipsis;white-space:nowrap}.ms-file b{color:#6951b9;text-transform:capitalize}.ms-progress .progress{border-radius:12px!important;overflow:hidden!important}.ms-progress .progress-bar{background:#8057e8!important;border-radius:12px!important}
</style>
'''))


In [ ]:
"""Vertical Voilà interface. No analysis starts until a button is clicked."""
import ipywidgets as widgets
from ipyfilechooser import FileChooser

def default_model_file(name):
    """Use a model downloaded by MyoTools, when it is present."""
    import os
    folder = Path(os.environ.get("MYOTOOLS_MODEL_DIR", str(Path.cwd() / "Hub" / "runtime" / "models")))
    model = folder / name
    return model if model.is_file() else None


def default_image_directory():
    """Use the user's real Desktop, or their home if no Desktop is available.

    This standard-library-only function is embedded in the notebooks so they
    also work when launched directly, without importing or starting the hub.
    """
    import os
    import subprocess
    from pathlib import Path

    if os.name == "nt":
        command = ["powershell.exe", "-NoProfile", "-Command",
                   "[Console]::OutputEncoding = [System.Text.Encoding]::UTF8; [Environment]::GetFolderPath('DesktopDirectory')"]
    else:
        command = ["xdg-user-dir", "DESKTOP"]
    try:
        result = subprocess.run(command, capture_output=True, text=True, timeout=5,
                                encoding="utf-8", errors="replace")
        if result.returncode == 0 and result.stdout.strip():
            desktop = Path(result.stdout.strip().lstrip("\ufeff")).expanduser()
            if desktop.is_absolute() and desktop.is_dir():
                return desktop
    except (OSError, subprocess.SubprocessError):
        pass
    home = Path.home()
    return next((path for path in (home / "Desktop", home / "Bureau") if path.is_dir()), home)

from IPython.display import display, HTML


def label(title, description=""):
    return widgets.HTML(value=f'<div class="ms-field-label"><strong>{html.escape(title)}</strong>'
                              f'<span>{html.escape(description)}</span></div>')


def section(number, title, description, children):
    heading = widgets.HTML(value=f'<div class="ms-section-heading"><span class="ms-number">{number}</span>'
                                  f'<div><h2>{html.escape(title)}</h2><p>{html.escape(description)}</p></div></div>')
    card = widgets.VBox([heading, *children], layout=widgets.Layout(width="100%"))
    card.add_class("ms-card")
    return card


class MyoSegApp:
    def __init__(self):
        self.busy = False
        self.task = None
        self.stop_event = threading.Event()
        self.messages = []
        self.settings_widgets = []
        self.style = {"description_width": "205px"}
        self.control_layout = widgets.Layout(width="580px", max_width="100%")
        self.header = widgets.HTML(value=f'''
        <div class="ms-hero">
          <div class="ms-hero-top"><span class="ms-mark">MS</span><span class="ms-version">VERSION {APP_VERSION} · CELLPOSE 3</span></div>
          <h1>MyoSeg</h1>
          <p>Segment fluorescence or RGB H&amp;E skeletal muscle cross-sections.<br>Measure cross-sectional area and minimum Feret diameter in one workflow.</p>
          <div class="ms-hero-foot"><span>01 &nbsp;Prepare</span><span>02 &nbsp;Segment</span><span>03 &nbsp;Measure</span></div>
        </div>''')
        self.mode = self.control(widgets.Dropdown, "Workflow", "full", options=[
            ("Full pipeline: prepare → segment → measure", "full"),
            ("Prepare images only", "prepare"), ("Measure existing label masks", "metrics")])
        self.input_chooser = FileChooser(str(default_image_directory()), show_only_dirs=True, rows=5)
        installed_model = default_model_file("MyoSeg")
        self.model_chooser = FileChooser(str(installed_model.parent if installed_model else Path.cwd()),
                                         filename=installed_model.name if installed_model else "",
                                         select_default=bool(installed_model), rows=5)
        self.fluorescence_model_chooser = self.model_chooser
        he_model = default_model_file("MyoSeg_HE")
        self.he_model_chooser = FileChooser(str(he_model.parent if he_model else Path.cwd()),
                                            filename=he_model.name if he_model else "",
                                            select_default=bool(he_model), rows=5)
        self.output_chooser = FileChooser(str(Path.cwd()), show_only_dirs=True, rows=4)
        for chooser in (self.input_chooser, self.model_chooser, self.he_model_chooser, self.output_chooser):
            chooser.layout = widgets.Layout(width="100%")
            chooser.add_class("ms-chooser")
        self.output_path = widgets.Text(value="", placeholder="Automatic: <input folder>/MyoSeg_results",
                                        layout=widgets.Layout(width="100%"))
        self.settings_widgets.append(self.output_path)
        self.output_chooser.register_callback(lambda chooser: setattr(self.output_path, "value", chooser.selected or ""))
        self.recursive = self.check("Include subfolders", False)
        self.input_hint = widgets.HTML()
        files_card = section("01", "Choose files", "Select folders and a trained Cellpose 3 model.", [
            self.mode, self.input_hint, self.input_chooser,
            label("Cellpose 3 model", "MyoSeg for fluorescence, MyoSeg_HE for H&E. Download both in the Hub, then select the model for your image type. Not required for preparation or existing-mask measurements."),
            self.model_chooser, label("Output folder", "Leave blank to create MyoSeg_results inside the input folder. You can type a new folder path."),
            self.output_path, self.output_chooser, self.recursive])

        self.files_card = files_card
        self.image_format = self.control(widgets.Dropdown, "Image type", "single", options=[
            ("Fluorescence / single-channel (original workflow)", "single"),
            ("H&E brightfield / original RGB image", "rgb")])
        self.channel = self.control(widgets.BoundedIntText, "Fluorescence channel index", 3, min=0, max=999)
        self.axes = self.control(widgets.Dropdown, "TIFF channel layout", "auto", options=[
            ("Auto (use TIFF/OME axes)", "auto"), ("Channels first: C × Y × X", "CYX"),
            ("Channels last: Y × X × C", "YXC")])
        self.all_scenes = self.check("Process all CZI scenes / TIFF series", True)
        self.time = self.control(widgets.BoundedIntText, "Time point", 0, min=0, max=99999)
        self.z = self.control(widgets.BoundedIntText, "Z plane", 0, min=0, max=99999)
        self.scale = self.control(widgets.BoundedFloatText, "Resize factor", 1., min=.01, max=10., step=.1)
        self.calibration = self.control(widgets.Dropdown, "Pixel calibration", "manual", options=[
            ("Manual (use value below)", "manual"), ("Image metadata, with manual fallback", "metadata")])
        self.pixel_size = self.control(widgets.BoundedFloatText, "Input pixel size (µm)", .6928, min=.000001, max=1e6, step=.01)
        self.normalize_nonzero = self.check("Normalize positive intensities. Keep zero background", True)
        self.calibration_hint = widgets.HTML()
        self.image_format_hint = widgets.HTML(value="")
        self.prep_widgets = [self.image_format, self.channel, self.axes, self.all_scenes, self.time, self.z, self.scale, self.normalize_nonzero]
        preparation_card = section("02", "Prepare images", "Fluorescence selects one 2D channel. H&E RGB preserves its original colors (3 samples per pixel). Time and Z indices start at 0.", [
            self.image_format, self.image_format_hint, self.channel, self.axes, self.all_scenes, self.time, self.z, self.scale,
            self.calibration, self.pixel_size, self.calibration_hint, self.normalize_nonzero])

        self.diameter = self.control(widgets.BoundedFloatText, "Fiber diameter (px)", 80., min=.1, max=1e6, step=5.)
        self.diameter_crop = self.control(widgets.Dropdown, "Diameter guide crop", 512, options=[256,384,512,768,1024])
        self.diameter_guide = self.button("Preview diameter guide", "circle-o", "secondary")
        self.diameter_guide_image = widgets.Image(format="png",layout=widgets.Layout(width="100%",max_width="520px"))
        self.diameter_guide_text = widgets.HTML(value='<div class="ms-note">A yellow circle shows the chosen diameter in pixels of the processed image.</div>')
        self.device = self.control(widgets.Dropdown, "Processing device", "auto", options=[
            ("Auto: CUDA GPU if available, otherwise CPU", "auto"), ("CPU", "cpu"), ("CUDA GPU (required)", "cuda")])
        self.rgb_model_channels = self.control(widgets.Dropdown, "RGB model input", "two", options=[
            ("Standard Cellpose 3: two selected channels", "two"),
            ("Custom model trained with all three RGB channels", "three")])
        self.cellpose_primary = self.control(widgets.Dropdown, "RGB Cellpose channel 1", 0, options=[
            ("Grayscale (combine RGB)", 0), ("Red", 1), ("Green", 2), ("Blue", 3)])
        self.cellpose_secondary = self.control(widgets.Dropdown, "RGB Cellpose channel 2", 0, options=[
            ("None (zero channel)", 0), ("Red", 1), ("Green", 2), ("Blue", 3)])
        self.cellpose_channel_hint = widgets.HTML(value='<div class="ms-note">Choose the <strong>same channel configuration used to train your custom Cellpose 3 model</strong>. Standard models use two channels: [0,0] converts RGB to grayscale + zeros, while e.g. [1,3] selects red and blue. Select three channels <em>only</em> if your model was explicitly trained with all RGB channels.</div>')
        self.flow_threshold = self.control(widgets.BoundedFloatText, "Flow threshold", .4, min=0., max=100., step=.1)
        self.cellprob_threshold = self.control(widgets.FloatText, "Cell probability threshold", 0.)
        self.min_size = self.control(widgets.BoundedIntText, "Minimum size (pixels)", 15, min=0, max=100000000)
        self.batch_size = self.control(widgets.BoundedIntText, "Cellpose batch size", 8, min=1, max=1024)
        self.cellpose_normalize = self.check("Use Cellpose percentile normalization", True)
        self.invert = self.check("Invert intensities before segmentation", False)
        self.seg_widgets = [self.diameter, self.device, self.rgb_model_channels, self.cellpose_primary, self.cellpose_secondary,
                            self.flow_threshold, self.cellprob_threshold,
                            self.min_size, self.batch_size, self.cellpose_normalize, self.invert]
        segmentation_card = section("03", "Segment muscle fibers", "Diameter and minimum size refer to the resized image (px and px²). For H&E, use parameters appropriate to the training images, then test one image.", [
            *self.seg_widgets, self.cellpose_channel_hint, self.diameter_crop, self.diameter_guide, self.diameter_guide_text, self.diameter_guide_image,
            label("Thresholds", "Lower cell probability thresholds usually include more candidates. A stricter flow threshold rejects more inconsistent masks. Lower batch size if GPU memory is insufficient.")])

        self.compute_feret = self.check("Calculate minimum Feret diameter", True)
        self.exclude_border = self.check("Exclude fibers touching the image border", False)
        self.area_min = self.control(widgets.BoundedFloatText, "Minimum CSA (µm²)", 0., min=0., max=1e12, step=50.)
        self.area_max = self.control(widgets.BoundedFloatText, "Maximum CSA (0 = off)", 0., min=0., max=1e12, step=50.)
        self.csa_max = self.control(widgets.BoundedFloatText, "CSA distribution max", 6750., min=1., max=1e9, step=250.)
        self.csa_step = self.control(widgets.BoundedFloatText, "CSA bin width (µm²)", 250., min=.001, max=1e9, step=50.)
        self.feret_max = self.control(widgets.BoundedFloatText, "Feret distribution max", 100., min=.001, max=1e6, step=5.)
        self.feret_step = self.control(widgets.BoundedFloatText, "Feret bin width (µm)", 5., min=.001, max=1e6, step=1.)
        self.metric_widgets = [self.compute_feret, self.exclude_border, self.area_min, self.area_max,
                               self.csa_max, self.csa_step, self.feret_max, self.feret_step]
        metrics_card = section("04", "Measure fibers", "Filters apply to exported statistics, not to the saved label masks. All detected fibers remain in the detailed table with an inclusion flag.", [
            *self.metric_widgets, label("Distributions", "Values at or above the maximum go into an overflow bin. Percentages include every measured fiber. Minimum Feret uses the convex hull of pixel centers, as in the source notebook.")])

        self.save_previews = self.check("Save a boundary preview for each image", True)
        self.preview_size = self.control(widgets.BoundedIntText, "Preview max side (px)", 1400, min=200, max=6000, step=100)
        exports_card = section("05", "Save outputs", "Masks and prepared images retain full resolution after your chosen resize. Only PNG previews are reduced.", [
            self.save_previews, self.preview_size,
            widgets.HTML(value='<div class="ms-output-map"><strong>Prepared_images/</strong> selected grayscale plane or original RGB image, as OME-TIFF<br>'
                               '<strong>Masks/</strong> integer label TIFFs · <strong>Previews/</strong> fiber boundaries<br>'
                               '<strong>Tables/</strong> CSV summaries and per-image fiber measurements<br>'
                               '<strong>segmentation_results.xlsx</strong> summary, CSA/Feret distributions, detailed labels<br>'
                               '<strong>Log/</strong> settings, checkpoints, errors and workbook backups</div>')])

        self.preview_button = self.button("Preview input", "image", "secondary")
        self.test_button = self.button("Test first image", "flask", "secondary")
        self.run_button = self.button("Run / resume analysis", "play", "primary")
        self.stop_button = self.button("Stop after image", "stop", "secondary")
        self.stop_button.disabled = True
        self.force = self.check("Recompute existing outputs (ignore checkpoints)", False)
        buttons = widgets.HBox([self.preview_button, self.test_button, self.run_button, self.stop_button],
                               layout=widgets.Layout(width="100%", flex_flow="row wrap"))
        buttons.add_class("ms-buttons")
        self.status = widgets.HTML(value=self.status_html("ready", "Ready to analyze"))
        self.progress = widgets.FloatProgress(value=0, min=0, max=1,
                                              layout=widgets.Layout(width="100%", height="12px"))
        self.progress.add_class("ms-progress")
        self.progress_label = widgets.HTML()
        self.result = widgets.HTML()
        self.log = widgets.HTML()
        self.file_status = widgets.HTML(value="<div class=\"ms-note\">Processed files will appear here.</div>")
        self.file_states = {}
        self.preview_image = widgets.Image(format="png", layout=widgets.Layout(width="100%", max_width="100%", max_height="650px", display="none"))
        self.preview_image.add_class("ms-preview")
        self.preview_caption = widgets.HTML()
        help_html = widgets.HTML(value='''<div class="ms-help"><strong>Suggested workflow</strong><br>
            1. Choose fluorescence or H&amp;E RGB, then preview the image and check calibration.<br>
            2. Test the first image and inspect its saved mask.<br>
            3. Run the batch. Re-running checks saved stages and resumes valid outputs.<br>
            To edit images before segmentation, use Prepare images only. Then select Prepared_images as input,
            use resize factor 1, and choose image metadata calibration.<br>
            To measure manually corrected masks, choose Measure existing label masks and select their folder.<br>
            <strong>H&amp;E:</strong> Keep RGB colors unchanged and match the Cellpose channel selection to model training. Do not assume that an RGB-trained model accepts an arbitrarily selected individual color channel.</div>''')
        run_card = section("06", "Run analysis", "A one-image test writes real outputs. Stop keeps completed work; an individual image finishes before stopping. Failed images are listed in Tables/Errors.csv and the run log.", [
            self.force, buttons, self.status, self.progress, self.progress_label, self.file_status, self.result,
            self.log, self.preview_image, self.preview_caption, help_html])
        self.environment = widgets.HTML(value='<div class="ms-footer">MyoSeg · local processing · '
            f'Cellpose: {html.escape(installed_version("cellpose"))} · '
            f'Voilà: {html.escape(installed_version("voila"))}<br>')
        self.widget = widgets.VBox([self.header, files_card, preparation_card, segmentation_card,
                                   metrics_card, exports_card, run_card, self.environment],
                                  layout=widgets.Layout(width="100%"))
        self.widget.add_class("myoseg-app")
        self.mode.observe(self.refresh, names="value")
        self.image_format.observe(self.switch_model, names="value")
        self.rgb_model_channels.observe(self.refresh, names="value")
        for widget in (self.scale, self.pixel_size, self.calibration):
            widget.observe(self.refresh, names="value")
        self.diameter_guide.on_click(lambda _: self.show_diameter_guide())
        self.preview_button.on_click(lambda _: self.start("preview"))
        self.test_button.on_click(lambda _: self.start("test"))
        self.run_button.on_click(lambda _: self.start("run"))
        self.stop_button.on_click(self.request_stop)
        self.refresh()

    def control(self, cls, description, value, **kwargs):
        widget = cls(description=description, value=value, style=self.style,
                     layout=widgets.Layout(width="580px", max_width="100%"), **kwargs)
        self.settings_widgets.append(widget)
        return widget

    def check(self, description, value):
        widget = widgets.Checkbox(description=description, value=value, indent=False,
                                  layout=widgets.Layout(width="100%", min_width="0"))
        self.settings_widgets.append(widget)
        return widget

    @staticmethod
    def button(description, icon, style):
        widget = widgets.Button(description=description, icon=icon,
                                layout=widgets.Layout(width="215px", height="44px"))
        widget.add_class(f"ms-{style}-button")
        return widget

    @staticmethod
    def status_html(state, text):
        return f'<div class="ms-status ms-{state}"><span class="ms-status-dot"></span>{html.escape(text)}</div>'

    def switch_model(self, *_):
        chosen = self.he_model_chooser if self.image_format.value == "rgb" else self.fluorescence_model_chooser
        previous = self.model_chooser
        if chosen is not previous:
            self.model_chooser = chosen
            self.files_card.children = tuple(chosen if widget is previous else widget
                                             for widget in self.files_card.children)
        self.refresh()

    def refresh(self, *_):
        if self.busy:
            return
        for widget in self.settings_widgets:
            widget.disabled = False
        mode = self.mode.value
        for widget in self.prep_widgets:
            widget.disabled = mode == "metrics"
        for widget in self.seg_widgets:
            widget.disabled = mode != "full"
        rgb = self.image_format.value == "rgb"
        if rgb and mode != "metrics":
            self.channel.disabled = True
            self.normalize_nonzero.disabled = True
        if not rgb or mode != "full":
            self.rgb_model_channels.disabled = True
        if not rgb or mode != "full" or self.rgb_model_channels.value == "three":
            self.cellpose_primary.disabled = True
            self.cellpose_secondary.disabled = True
        self.image_format_hint.value = ('<div class="ms-note">RGB mode: preserve original red/green/blue values. '
                                        'The fluorescence channel index and positive-intensity normalization are ignored. '
                                        'Select Cellpose RGB channels below to match the model training.</div>'
                                        if rgb else '<div class="ms-note">Single-channel mode: select the fluorescence channel. '
                                        'RGB TIFFs are split by their R/G/B component in this mode (indices 0, 1, 2).</div>')
        for widget in self.metric_widgets:
            widget.disabled = mode == "prepare"
        self.input_hint.value = label("Input folder", "All TIFFs in this folder will be interpreted as integer label masks."
                                     if mode == "metrics" else "TIFF, OME-TIFF and CZI images. Select your image folder.").value
        self.test_button.description = "Test first mask" if mode == "metrics" else "Test first image"
        self.run_button.description = "Prepare images" if mode == "prepare" else "Run / resume analysis"
        if mode == "metrics":
            text = "Pixel size refers to the existing mask resolution. Masks are not resized."
        elif self.calibration.value == "manual":
            text = f"Approximate output calibration: {self.pixel_size.value / self.scale.value:.6g} µm/px. Exact X/Y values account for resize rounding."
        else:
            text = "Use image metadata where available; otherwise use the manual input pixel size. Resize is accounted for automatically."
        self.calibration_hint.value = f'<div class="ms-note">{html.escape(text)}</div>'

    def get_settings(self):
        selected = self.input_chooser.selected
        if not selected:
            raise ValueError("Select an input folder using its Select button.")
        output = self.output_path.value.strip() or str(Path(selected) / "MyoSeg_results")
        settings = {key: getattr(self, key).value for key in (
            "mode", "recursive", "image_format", "channel", "axes", "all_scenes", "time", "z", "scale",
            "calibration", "pixel_size", "normalize_nonzero", "diameter", "device",
            "rgb_model_channels", "cellpose_primary", "cellpose_secondary",
            "flow_threshold", "cellprob_threshold", "min_size", "batch_size", "cellpose_normalize",
            "invert", "compute_feret", "exclude_border", "area_min", "area_max", "csa_max",
            "csa_step", "feret_max", "feret_step", "save_previews", "preview_size", "force")}
        settings.update({"input_dir": selected, "output_dir": output,
                         "model": self.model_chooser.selected or ""})
        return settings

    def show_diameter_guide(self):
        try:
            s = validate_settings(self.get_settings(),require_model=False)
            if s["mode"] == "metrics":
                raise ValueError("Diameter guide requires a source image, not existing masks.")
            path=discover_files(s)[0]
            plane,info=load_plane(path,0,s)
            # Crop before resizing where possible; this guide is visual only.
            prepared,info=prepare_plane(plane,info,s)
            height,width=prepared.shape[:2]
            side=int(self.diameter_crop.value)
            cy,cx=height//2,width//2
            y0=max(0,min(height-side,cy-side//2)); x0=max(0,min(width-side,cx-side//2))
            crop=prepared[y0:y0+side,x0:x0+side]
            from PIL import ImageDraw
            from io import BytesIO
            arr=np.asarray(crop)
            if arr.ndim == 3 and arr.shape[-1]==3:
                img=Image.fromarray(arr.astype(np.uint8)).convert("RGB")
            else:
                arr=arr.astype(np.float32)
                lo,hi=np.percentile(arr[np.isfinite(arr)],[1,99.5]) if np.isfinite(arr).any() else (0.,1.)
                gray=np.uint8(np.clip((arr-lo)/max(hi-lo,1.e-6),0,1)*255)
                img=Image.fromarray(gray).convert("RGB")
            d=float(self.diameter.value); x,y=img.size[0]//2,img.size[1]//2
            ImageDraw.Draw(img).ellipse((x-d/2,y-d/2,x+d/2,y+d/2),outline=(255,192,45),width=3)
            img.thumbnail((520,520));buf=BytesIO();img.save(buf,format="PNG")
            self.diameter_guide_image.value=buf.getvalue()
            self.diameter_guide_text.value=f'<div class="ms-note">{html.escape(path.name)} · Guide {d:g} px at processed-image resolution. Compare against representative fibers.</div>'
        except Exception as exc:
            self.diameter_guide_text.value=f'<div class="ms-note">{html.escape(str(exc))}</div>'

    def on_event(self, event):
        kind = event["kind"]
        if kind == "log":
            self.messages.append(event["text"])
            self.messages = self.messages[-12:]
            self.log.value = '<div class="ms-log">' + ''.join(
                f'<div>{html.escape(line)}</div>' for line in self.messages) + '</div>'
        elif kind == "file":
            self.file_states[event["file"]] = event["state"]
            rows=list(self.file_states.items())[-150:]
            self.file_status.value="<div class=\"ms-files\">"+"".join(f"<div class=\"ms-file\"><span>{html.escape(n)}</span><b>{html.escape(st)}</b></div>" for n,st in reversed(rows))+"</div>"
        elif kind == "progress":
            self.progress.value = event["current"] / max(1, event["total"])
            self.progress_label.value = f'<div class="ms-note">{html.escape(event["text"])}</div>'
        elif kind == "preview":
            self.preview_image.value = event["png"]
            self.preview_image.layout.display = "block"
            self.preview_caption.value = f'<div class="ms-note">{html.escape(event["text"])}</div>'

    def start(self, action):
        if self.busy:
            return
        try:
            settings = validate_settings(self.get_settings(), require_model=action != "preview")
            loop = asyncio.get_running_loop()
        except Exception as exc:
            self.status.value = self.status_html("error", str(exc))
            return
        self.busy = True
        self.stop_event.clear()
        self.messages.clear()
        self.file_states.clear()
        self.file_status.value="<div class=\"ms-note\">Waiting for files...</div>"
        self.log.value = ""
        self.result.value = ""
        self.progress.value = 0
        for widget in self.settings_widgets:
            widget.disabled = True
        for button in (self.preview_button, self.test_button, self.run_button):
            button.disabled = True
        self.stop_button.disabled = action == "preview"
        self.status.value = self.status_html("running", "Preparing input preview…" if action == "preview" else "Analysis running…")
        self.task = loop.create_task(self.execute(action, settings, loop))

    async def execute(self, action, settings, loop):
        try:
            if action == "preview":
                event = await loop.run_in_executor(None, lambda: preview_input(settings))
                self.on_event(event)
                self.progress.value = 1
                self.status.value = self.status_html("complete", "Preview ready. Check the channel and calibration before segmentation.")
            else:
                def emit(event):
                    loop.call_soon_threadsafe(self.on_event, event)
                result = await loop.run_in_executor(None, lambda: run_pipeline(
                    settings, emit=emit, stop_event=self.stop_event, first_only=action == "test"))
                state = "error" if result["errors"] else "complete"
                text = ("Stopped after saving completed work" if result["stopped"] else "Run finished")
                self.status.value = self.status_html(state, f"{text} · {result['successful']} successful · {result['errors']} error(s)")
                self.result.value = '<div class="ms-note"><strong>Outputs</strong><br>' + html.escape(result["output_dir"]) + (
                    '<br><strong>Workbook:</strong> segmentation_results.xlsx' if result["workbook"] else '') + '</div>'
        except Exception as exc:
            self.status.value = self.status_html("error", str(exc))
            self.on_event({"kind": "log", "text": f"ERROR: {exc}"})
        finally:
            self.busy = False
            self.stop_button.disabled = True
            for button in (self.preview_button, self.test_button, self.run_button):
                button.disabled = False
            self.refresh()

    def request_stop(self, _):
        if self.busy:
            self.stop_event.set()
            self.stop_button.disabled = True
            self.status.value = self.status_html("running", "Stop requested. Finishing the current image and saving results…")


myoseg_app = MyoSegApp()
display(myoseg_app.widget)
